# Affordable Cambridge: housing data behind the website

The [Affordable Cambridge](https://affordablecambridge.org/) website credits three figures to "A Better Cambridge housing data." This notebook shows where each one comes from:

1. **−3 and +1,736:** net homes permitted by neighborhood, 2018–2025
2. **−67:** homes lost in West Cambridge to downconversions, across 56 developments, 1996–2024
3. **Home value:** typical Cambridge home value, January of each year, 2000–2026

All three use public data. The website's other figures link straight to their own sources.

In [1]:
from pathlib import Path
import pandas as pd

DATA_RAW = Path("../data/raw")
STARTS_CSV = DATA_RAW / "cambridge_housing_starts_2026-11-15.csv"
ZHVI_CSV = DATA_RAW / "City_zhvi_uc_sfrcondo_tier_0.33_0.67_sm_sa_month.csv"

starts = pd.read_csv(STARTS_CSV)
starts["Neighborhood"] = starts["Neighborhood"].str.replace(r"^\d+\s*-\s*", "", regex=True)

## 1. Net homes permitted by neighborhood, 2018–2025

**Source:** City of Cambridge, ["Housing Starts 1996-Present"](https://data.cambridgema.gov/d/6nxj-uqfe), Open Data Portal, downloaded November 15, 2026 (`cambridge_housing_starts_2026-11-15.csv`). A download from November 23, 2026 has the same rows.

**Method:** for each neighborhood, add up the net change in homes across every permit from 2018 through 2025. A permit that replaces a two-family house with a six-unit building counts as +4.

We leave out one 2025 demolition, 221 Mt Auburn St (the Riverview Apartments, −63 homes) in West Cambridge. It was a one-time private demolition unrelated to housing policy, and including it would hide the pattern in the rest of West Cambridge. With it, West Cambridge is −66.

**Earlier figures.** The website used to show −4 for West Cambridge and +1,690 for East Cambridge. Those came from a hand-curated copy of the city's data made in May 2026 (`cambridge_housing_permits_curated.csv`), which was missing permits the city added later, such as 49 Sixth St in East Cambridge (+46) and five in West Cambridge (13 Saville St, 141 and 69 Fayerweather St, 194 Lakeview Ave, 252 Lexington Ave).

In [2]:
MT_AUBURN = "221 Mt Auburn St"
assert (starts["Application Address"] == MT_AUBURN).sum() == 1

in_window = starts["Year Permitted"].between(2018, 2025)
net = (starts[in_window & (starts["Application Address"] != MT_AUBURN)]
       .groupby("Neighborhood")["Net Change in Units"].sum().astype(int)
       .sort_values(ascending=False).rename("Net homes, 2018–2025"))
net.to_frame()

,"Net homes, 2018–2025"
Neighborhood,
East Cambridge,1736
North Cambridge,835
Cambridge Highlands,676
The Port,339
Area 2/MIT,163
Neighborhood Nine,117
Baldwin,90
Wellington-Harrington,26
Cambridgeport,18


## 2. West Cambridge: homes lost to downconversions, 1996–2024

**Source:** City of Cambridge, ["Housing Starts 1996-Present"](https://data.cambridgema.gov/d/6nxj-uqfe), downloaded November 15, 2026 (`cambridge_housing_starts_2026-11-15.csv`). This is the same file as section 1, and the same data behind the website's Downconversions page.

**Method:** a downconversion is a permit that reduces the number of homes at an address while at least one home remains. Each one loses its homes before minus its homes after: a three-family house turned into a single-family house loses two homes. Demolitions that leave no homes are not counted. We use the same two rules as the Downconversions page:

- Leave out buildings that had 10 or more homes before. These are apartment buildings, not house conversions. In West Cambridge this drops 1-7 Brattle Circle (10 to 7) and 14 Concord Ave (47 to 46).
- Leave out a downconversion that a later permit reversed. In West Cambridge this drops 204 Fayerweather St, which went from 2 homes to 1 in 2022 and back to 2 in 2025.

**Single-family conversions.** The website used to show −46, which counted only buildings converted to a single home, using the older hand-curated copy described in section 1. With the city's current file and the rules above, that single-family subset is 43 developments and 47 homes (the last lines below). The extra home comes from the curated copy's missing and corrected records.

In [3]:
BEFORE, AFTER = "Prior Residential Units", "Permitted Residential Units"

def reversed_later(row):
    """True if a later permit at the same address added homes back."""
    later = starts[(starts["Application Address"] == row["Application Address"])
                   & (starts["Year Permitted"] > row["Year Permitted"])]
    return (later[AFTER] > later[BEFORE]).any()

wc = starts[(starts["Neighborhood"] == "West Cambridge")
            & starts["Year Permitted"].between(1996, 2024)
            & (starts[AFTER] < starts[BEFORE])
            & (starts[AFTER] >= 1)
            & (starts[BEFORE] < 10)].copy()
wc = wc[~wc.apply(reversed_later, axis=1)]
wc[[BEFORE, AFTER, "Year Permitted"]] = wc[[BEFORE, AFTER, "Year Permitted"]].astype(int)
wc["Homes lost"] = wc[BEFORE] - wc[AFTER]
wc = wc[["Application Address", "Year Permitted", BEFORE, AFTER, "Homes lost"]].sort_values(["Year Permitted", "Application Address"]).reset_index(drop=True)

sfh = wc[wc[AFTER] == 1]
print(f"{len(wc)} developments, {wc['Homes lost'].sum()} homes lost")
print(f"Of these, {len(sfh)} became single-family homes, losing {sfh['Homes lost'].sum()} homes")
wc

56 developments, 67 homes lost
Of these, 43 became single-family homes, losing 47 homes


,Application Address,Year Permitted,Prior Residential Units,Permitted Residential Units,Homes lost
0,180-182 Appleton St,1997,3,2,1
1,263-265 Mt Auburn St,1997,2,1,1
2,22 Garden St,1998,3,2,1
3,24 Garden St,1998,6,3,3
4,26-28 Garden St,1998,6,3,3
5,74 Fayerweather St,2000,2,1,1
6,167 Brattle St,2003,2,1,1
7,23 Gurney St,2003,2,1,1
8,68-70 Lakeview Ave.,2003,2,1,1
9,248-250 Concord Ave,2004,3,2,1


## 3. Typical Cambridge home value, 2000–2026

**Source:** [Zillow Research](https://www.zillow.com/research/data/), Zillow Home Value Index (ZHVI) for Cambridge, MA: single-family homes and condos, middle price tier (homes between the 33rd and 67th percentile), smoothed and seasonally adjusted. Values are in nominal dollars, not adjusted for inflation.

**Method:** take the January value for each year.

Zillow revises past values each month, so a newer download may differ slightly. The copy here was downloaded in May 2026.

In [4]:
zhvi = pd.read_csv(ZHVI_CSV)
cambridge = zhvi[(zhvi.RegionName == "Cambridge") & (zhvi.State == "MA")].iloc[0]

home_value = pd.DataFrame({"Year": range(2000, 2027)})
home_value["Typical home value"] = [round(cambridge[f"{y}-01-31"], -3) for y in home_value.Year]
home_value["Typical home value"] = home_value["Typical home value"].map("${:,.0f}".format)

ratio = cambridge["2026-01-31"] / cambridge["2000-01-31"]
print(f"January 2026 is {ratio:.1f} times January 2000")
home_value

January 2026 is 3.3 times January 2000


,Year,Typical home value
0,2000,"$314,000"
1,2001,"$373,000"
2,2002,"$421,000"
3,2003,"$455,000"
4,2004,"$473,000"
5,2005,"$505,000"
6,2006,"$517,000"
7,2007,"$494,000"
8,2008,"$491,000"
9,2009,"$479,000"
